# Week 11: Data Acquisition

By the end of this worksheet you will be able to

- Understand the fundamentals of Application Programming Interfaces (APIs) and how they facilitate the access and retrieval of data.  
- Use APIs to acquire data from the web.  
- Perform web scraping to extract data from the web.  
- Implement computer code to read in multi-file data sets.  
- Describe ethical considerations when acquiring data from the web  


## Getting Started

Before beginning the worksheet, let us load the necessary packages we will be using throughout the worksheet

In [ ]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(httr))
suppressPackageStartupMessages(library(jsonlite))
suppressPackageStartupMessages(library(rvest))
suppressPackageStartupMessages(library(arrow))

## Data Acquisition 
 Until this point, we were loading data locally using functions like `read_csv()`. In this worksheet, you will be collecting data from various sources on internet using the following methods:
- **APIs (Application Programming Interfaces):** Allow programmatic access to data from web services in structured formats like JSON or XML.
- **Web Scraping:** Extracts data directly from web pages when APIs are unavailable.
- **Parquet Files:** A columnar storage file format optimized for large-scale data processing.

### PART 1: API

There are two parts to making an API request:
- The resource you want to request (i.e., the data)  
- The verb specifying what you want to do with that resource (get it, update it, create one etc.)

The **resource** can be accessed using a URL or web address.

The base URL is the domain and part of the path that is included on all resources. It acts as the “root” for any particular resource.
For example, the GitHub API has a base URL of https://api.github.com/.

An endpoint specifies which resource on that domain you want to access.
For example, a specific user or organization on GitHub such as https://api.github.com/users/katieburak.

We will be using the `httr` package in R, which has functions for the following most important http verbs:

`GET()`: Return a representation of the current state of the resource  
`HEAD()`: Return the header of the resource  
`PATCH()`: Update a portion of the resource’s state  
`PUT()`: Update the resource to have a new state  
`DELETE()`: Remove the resource  
`POST()`: Add a new subresource  

We will be using the **Frankfurter API** which is a free, open-source API that provides current and historical foreign exchange rates published by the European Central Bank. You can find more information about this API on their website (https://frankfurter.dev/). An example of extracting the current conversion rate of USD to EUR:

In [ ]:
url <- "https://api.frankfurter.app/latest?from=USD&to=EUR" # Resource URL
response <- GET(url) # Make a GET request to the resource
data <- response |> content(as = "text") |> fromJSON() # Parsing the JSON response
data

The output of `fromJSON()` is a list with four elements: `amount`, `base`, `date`, and `rates`. Note that `rates` is itself a list. In R, lists can contain other lists as elements!

Let's extract the current conversion rate of USD to EUR from `data`.

In [ ]:
rates <- data |> as_tibble() |> pull(rates) |> as_tibble() |> pull(EUR) # Converting parsed data into a tibble
rates

The code above uses `as_tibble()` twice to convert lists into tibbles. This is because `rates` is itself a list nested inside `data`. The `pull()` function then extracts the desired elements.

In this particular example, since we are extracting a single value, we could have used data$rates$EUR directly, without converting the lists into tibbles. 

Let's break down the URL from the above code chunk:

- `https://api.frankfurter.app/latest` is the base URL and endpoint for the latest rates.  
- `?from=USD&to=EUR` are query parameters specifying the base currency (USD) and the target currency (EUR).

Other query parameters include:

- `from` - Base currency (3-letter ISO code)	
- `to` - Target currency or list of currencies (comma-separated)	
- `amount` - Amount of the base currency to convert	
- `start_date..end_date` - Range for historical time-series data (used in place of `/latest` endpoint)	


As an example, let's change the endpoint from **latest currency rates** to **all the available currencies**

In [ ]:
url <- "https://api.frankfurter.app/currencies" # Resource URL with currencies as endpoint
response <- GET(url)
data <- response |> 
        content(as = "text") |> 
        fromJSON() |> 
        enframe(name = "ISO Code", value = "Currency") |> # Convert the named list into a tibble
        unnest(cols = Currency)

head(data)

**Question 1**
<br> (1 point)

What were the exchange rates from CAD to EUR, GBP, and USD on January 15, 2024?

Using the API, programmatically retrieve the exchange rates and organize them into a tibble with 1 row and 3 columns, one for each currency. Do not manually enter the exchange rate values.

> **HINT**: Here is the start of the url for that particular date (`https://api.frankfurter.app/2024-01-15`), but you will have to change the url endpoint based on the exchange rates you want, separating currencies with commas (e.g., `USD,EUR,GBP`).

*Assign the table to variable `answer1`.*

In [ ]:
# answer1<- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer1

In [ ]:
library(digest)
stopifnot("answer1 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer1)), "9723c")), "80b089e8383b4048725cd865313c891a"))
stopifnot("dimensions of answer1 are not correct"= setequal(digest(paste(toString(dim(answer1)), "9723c")), "711faa6780c0969c69c60c6393ecaa3c"))
stopifnot("column names of answer1 are not correct"= setequal(digest(paste(toString(sort(colnames(answer1))), "9723c")), "1f08d749310f5e28a31b96b24dde17d0"))
stopifnot("types of columns in answer1 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer1, class)))), "9723c")), "b95f8d013a5fd47a76cef05ddb286ffb"))
stopifnot("values in one or more numerical columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.numeric))) sort(round(sapply(answer1[, sapply(answer1, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "9723c")), "a190e01b9f058d903af645a65c821f22"))
stopifnot("values in one or more character columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.character))) sum(sapply(answer1[sapply(answer1, is.character)], function(x) length(unique(x)))) else 0), "9723c")), "98d3ed422aa43034ed9a24d8ee22aeb6"))
stopifnot("values in one or more factor columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.factor))) sum(sapply(answer1[, sapply(answer1, is.factor)], function(col) length(unique(col)))) else 0), "9723c")), "98d3ed422aa43034ed9a24d8ee22aeb6"))

print('Success!')


Now let's retrieve data from January 2-31 2024 for USD to EUR conversion rates.

In [ ]:
start_date <- "2024-01-02"
end_date <- "2024-01-31"
url <- paste0("https://api.frankfurter.app/", start_date, "..", end_date, "?from=USD&to=EUR")
url ## check the url

In [ ]:
response <- GET(url)
data <- response |> content(as = "text") |> fromJSON()
rates_tibble <- data |> as_tibble() |> pull(rates) |> 
                    enframe(name = "date", value = "rates") #|>  # Convert the named list into a tibble
                   # unnest_wider(rates)

In [ ]:
response <- GET(url)
data <- response |> content(as = "text") |> fromJSON()
rates_tibble <- data |> as_tibble() |> pull(rates) |> 
                    enframe(name = "date", value = "rates") |>  # Convert the named list into a tibble
                    unnest_wider(rates) # Expand the nested lists into columns (only EUR in this case)

colnames(rates_tibble) <- c("date", "exchange_rate")
rates_tibble <- rates_tibble|> mutate(date = as.Date(date))

In [ ]:
rates_tibble

**Question 2** 
<br> (1 point)

Plot the trend of the above rates as a line plot. 

*Assign the answer to variable `answer2`.* 

In [ ]:
#answer2 <- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer2

In [ ]:
library(digest)
stopifnot("type of plot is not correct (if you are using two types of geoms, try flipping the order of the geom objects!)"= setequal(digest(paste(toString(sapply(seq_len(length(answer2$layers)), function(i) {c(class(answer2$layers[[i]]$geom))[1]})), "387a5")), "47d82841fe5593bfd2c1468afef8087f"))
stopifnot("variable x is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer2$layers)), function(i) {rlang::get_expr(c(answer2$layers[[i]]$mapping, answer2$mapping)$x)}), as.character))), "387a5")), "f7952961db1b111a84c957b29ed40d77"))
stopifnot("variable y is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer2$layers)), function(i) {rlang::get_expr(c(answer2$layers[[i]]$mapping, answer2$mapping)$y)}), as.character))), "387a5")), "04d71e800729f8c0d323cf821ed6b744"))
stopifnot("x-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$x)!= answer2$labels$x), "387a5")), "2a5cc36f598c9e7d128419846402716a"))
stopifnot("y-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$y)!= answer2$labels$y), "387a5")), "2a5cc36f598c9e7d128419846402716a"))
stopifnot("incorrect colour variable in answer2, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour)), "387a5")), "5945a82f4982dff104cdb9287f320c5f"))
stopifnot("incorrect shape variable in answer2, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$shape)), "387a5")), "5945a82f4982dff104cdb9287f320c5f"))
stopifnot("the colour label in answer2 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour) != answer2$labels$colour), "387a5")), "5945a82f4982dff104cdb9287f320c5f"))
stopifnot("the shape label in answer2 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour) != answer2$labels$shape), "387a5")), "5945a82f4982dff104cdb9287f320c5f"))
stopifnot("fill variable in answer2 is not correct"= setequal(digest(paste(toString(quo_name(answer2$mapping$fill)), "387a5")), "182b92b03ee9f615bf26e0dc661f3d8c"))
stopifnot("fill label in answer2 is not informative"= setequal(digest(paste(toString((quo_name(answer2$mapping$fill) != answer2$labels$fill)), "387a5")), "5945a82f4982dff104cdb9287f320c5f"))
stopifnot("position argument in answer2 is not correct"= setequal(digest(paste(toString(class(answer2$layers[[1]]$position)[1]), "387a5")), "68dca3202a091ce206bbed9c436f633d"))

print('Success!')

**Question 3**
<br> (1 point)

Which of the following best describes the overall trend in the line plot? 
 
  A) Increasing  
  B) Decreasing  
  C) Stable  

*Assign your quoted answer to `answer3`. Your answer should be one of "A", "B", or "C", surrounded by quotes.* 

In [ ]:
#answer3 <- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

In [ ]:
library(digest)
stopifnot("type of answer3 is not character"= setequal(digest(paste(toString(class(answer3)), "40809")), "921d9cd70b617fe97849740ec32ec423"))
stopifnot("length of answer3 is not correct"= setequal(digest(paste(toString(length(answer3)), "40809")), "abb01fb6fffc5226ae4a74e1cb475d26"))
stopifnot("value of answer3 is not correct"= setequal(digest(paste(toString(tolower(answer3)), "40809")), "b283caec173c88761d0c007882902be1"))
stopifnot("letters in string value of answer3 are correct but case is not correct"= setequal(digest(paste(toString(answer3), "40809")), "f7d257bb49d855f280b005619a011838"))

print('Success!')

**Question 4**
<br> (1 point)

Say you went on a work trip to France on "2024-06-12". You need to submit an expense report on how much you spent there in CAD for your reimbursements and all your receipts show the spending in EUR to a total of 260 EUR. 

Use the Frankfurter API to convert this EUR amount to CAD with the conversion rate at the day of expense. 

*Assign the CAD value to the variable `answer4`.*

In [ ]:
#answer4 <- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer4

In [ ]:
library(digest)
stopifnot("type of answer4 is not numeric"= setequal(digest(paste(toString(class(answer4)), "1df20")), "081b19dbd8dc298e9bbfc1b8ea714008"))
stopifnot("value of answer4 is not correct (rounded to 2 decimal places)"= setequal(digest(paste(toString(round(answer4, 2)), "1df20")), "f37704edf08c17bff581593a27808aad"))
stopifnot("length of answer4 is not correct"= setequal(digest(paste(toString(length(answer4)), "1df20")), "3d1d07278ce57fd36f4ba7db77fe3875"))
stopifnot("values of answer4 are not correct"= setequal(digest(paste(toString(sort(round(answer4, 2))), "1df20")), "f37704edf08c17bff581593a27808aad"))

print('Success!')

**Question 5**
<br> (1 point)

Analyze the volatility of the Canadian Dollar (CAD) against USD, EUR, and GBP during the COVID-19 pandemic (March 2020 to May 2023). 

Plot the trend of CAD conversion rates to the three currencies over the time period.

> **HINT**: you may need to pivot your data into long format.

*Replace the ... in the code chunk below with appropriate code. Uncomment the lines corresponding to code and run it. Assign the ggplot object to `answer5`.* 

In [ ]:
# url <- "..."

# response <- ...(url)
# data <- ... |> ...(as = "...") |> ...()

# rates_df <- ... |> ... |> 
#             ... |> ...(name = "date", value = "rates") |> ...(rates)

# rates_df <- rates_df |> ...(date = as.Date(date))

# df_long <- rates_df |> ...(cols = -date, names_to = "currency", values_to = "rates")

# answer5 <- df_long|> 
#   ...(aes(x = ..., y = ..., color = currency)) +
#   ...() +
#   labs(
#     title = "CAD Exchange Rate Volatility (Mar 2020 – May 2023)",
#     x = "Date",
#     y = "Exchange Rate",
#     color = "Currency"
#   ) +
#   theme_minimal()

# YOUR CODE HERE
fail()

answer5

In [ ]:
library(digest)
stopifnot("type of plot is not correct (if you are using two types of geoms, try flipping the order of the geom objects!)"= setequal(digest(paste(toString(sapply(seq_len(length(answer5$layers)), function(i) {c(class(answer5$layers[[i]]$geom))[1]})), "a1a2d")), "f959cf74ad1c4ea4b4051442dd10b22c"))
stopifnot("variable x is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer5$layers)), function(i) {rlang::get_expr(c(answer5$layers[[i]]$mapping, answer5$mapping)$x)}), as.character))), "a1a2d")), "653d5ade7cd0febc4edf006152a9b896"))
stopifnot("variable y is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer5$layers)), function(i) {rlang::get_expr(c(answer5$layers[[i]]$mapping, answer5$mapping)$y)}), as.character))), "a1a2d")), "529239e5b176419ee8e9c618271bad1c"))
stopifnot("x-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$x)!= answer5$labels$x), "a1a2d")), "fc8964ae0acb9b3bf079fbfb04d03fc4"))
stopifnot("y-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$y)!= answer5$labels$y), "a1a2d")), "fc8964ae0acb9b3bf079fbfb04d03fc4"))
stopifnot("incorrect colour variable in answer5, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour)), "a1a2d")), "798cb97cd7435b5fbda6752f0c3598fb"))
stopifnot("incorrect shape variable in answer5, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$shape)), "a1a2d")), "13af0d2d602eabb8f1ad572f93636176"))
stopifnot("the colour label in answer5 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour) != answer5$labels$colour), "a1a2d")), "fc8964ae0acb9b3bf079fbfb04d03fc4"))
stopifnot("the shape label in answer5 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour) != answer5$labels$shape), "a1a2d")), "13af0d2d602eabb8f1ad572f93636176"))
stopifnot("fill variable in answer5 is not correct"= setequal(digest(paste(toString(quo_name(answer5$mapping$fill)), "a1a2d")), "6924a270250e5559b44d90f57dcd2fd1"))
stopifnot("fill label in answer5 is not informative"= setequal(digest(paste(toString((quo_name(answer5$mapping$fill) != answer5$labels$fill)), "a1a2d")), "13af0d2d602eabb8f1ad572f93636176"))
stopifnot("position argument in answer5 is not correct"= setequal(digest(paste(toString(class(answer5$layers[[1]]$position)[1]), "a1a2d")), "ebd2c59c3bb6285839f461574904087b"))

print('Success!')

### PART 2: Web Scraping

Web scraping involves automatically extracting data from websites. Unlike APIs, which provide structured access to data, scraping involves pulling data directly from HTML content. Web scraping is commonly used to extract data from online sources when no API available to facilitate data acquisition. It is very important that before scraping any website, we check its terms of service or robots.txt file to make sure web scraping is allowed for that webpage.

For this part, we will be using the `rvest` package in R to extract data from HTML. Using `rvest` for web scraping requires a basic understanding of HTML including the structure of HTML content (elements, attributes). `rvest` functions make it easier to access the required data from HTML content and some of the commonly used functions include: 

- `read_html()`: Loads HTML content from a webpage URL or a character string.  

- `html_element()`: Retrieves a single matching element based on a CSS selector.  

- `html_elements()`: Retrieves all matching elements using CSS selectors.  

- `html_table()`: Converts HTML tables into data frames.  

- `html_text()/html_text2()`: Gets the text content from within HTML tags.  

- `html_name()`: Returns the tag name(s) of HTML elements.  

- `html_attr()`: Retrieves a single attribute.  

- `html_attrs()`: Retrieves all attributes.  

Let's get back to the Frankfurter exchange rate data. Suppose you don’t have access to the API, but someone has embedded a historical CAD exchange rate table on a webpage. Instead of downloading a CSV, we want to extract the table directly using web scraping.

**Question 6**
<br> (1 point)

Using the CSS selectors, scrape only the headers of the table from the webpage at the URL provided below. Use `read_html()` to load the webpage's HTML content into an object named `page`.

> **HINT**: You can use the SelectorGadget Chrome extension to identify the appropriate CSS selectors and then use `html_elements()` to extract the elements. 

The extracted HTML elements need to be further processed using a parser function such as `html_text()` or `html_text2()` to make the output readable. 

*Replace the ... in the code chunk below with appropriate code. Uncomment the lines corresponding to code and run it. Assign the parsed output to variable `answer6`.* 

In [ ]:
# "FILL IN YOUR ANSWER HERE"

# url <-"https://ubc-dsci.github.io/dsci-200/schedule/data/currency-data.html"
# page <- read_html(...)

# answer6 <- ... |> 
#   ...("...") |> 
#   ...()

# YOUR CODE HERE
fail()

answer6

In [ ]:
library(digest)
stopifnot("type of answer6 is not character"= setequal(digest(paste(toString(class(answer6)), "6823a")), "257867856eabe6a3ceda445ba95c7e66"))
stopifnot("length of answer6 is not correct"= setequal(digest(paste(toString(length(answer6)), "6823a")), "61171f8899167c4c3f5f975d37961b75"))
stopifnot("value of answer6 is not correct"= setequal(digest(paste(toString(tolower(answer6)), "6823a")), "0eea3eee04938523f65149d7cdf98ee4"))
stopifnot("letters in string value of answer6 are correct but case is not correct"= setequal(digest(paste(toString(answer6), "6823a")), "9811f479a179bf7d89d7872c4524001e"))

print('Success!')

**Question 7** 
<br> (1 point)

Using the CSS selectors, scrape only the `date` column from the table and parse it to a readable format. 

> **HINT**: Is there a CSS selector that can select the first cell in any table row? 

*Assign the parsed output to variable `answer7`.*

In [ ]:
# answer7 <- "FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer7)

In [ ]:
library(digest)
stopifnot("type of answer7 is not character"= setequal(digest(paste(toString(class(answer7)), "67707")), "fe96ed4884e5c52dabd605330846ebb2"))
stopifnot("length of answer7 is not correct"= setequal(digest(paste(toString(length(answer7)), "67707")), "199e52a67f64e4628c34b21c1f0cc5cd"))
stopifnot("value of answer7 is not correct"= setequal(digest(paste(toString(tolower(answer7)), "67707")), "4184298fdc87b4f2bdd99290730b72d7"))
stopifnot("letters in string value of answer7 are correct but case is not correct"= setequal(digest(paste(toString(answer7), "67707")), "4184298fdc87b4f2bdd99290730b72d7"))

print('Success!')

**Question 8**
<br> (1 point)

Extract the entire table from the HTML content using `html_table()` as a tibble. 

>**HINT**: Use `pluck()` to extract the tibble from `html_table()`.

*Assign the tibble to variable `answer8`.*

In [ ]:
# answer8 <- "FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer8)

In [ ]:
library(digest)
stopifnot("answer8 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer8)), "d9d10")), "b8c16bf1c2dc21b4a20cd9d42c55463a"))
stopifnot("dimensions of answer8 are not correct"= setequal(digest(paste(toString(dim(answer8)), "d9d10")), "e0c8bcc57f59f23c62976d2b6fb848e7"))
stopifnot("column names of answer8 are not correct"= setequal(digest(paste(toString(sort(colnames(answer8))), "d9d10")), "4472beceb9daab7ad3e9bb5c06fa10c4"))
stopifnot("types of columns in answer8 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer8, class)))), "d9d10")), "f078ef0cb975c6ce0e89b8625b13c900"))
stopifnot("values in one or more numerical columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.numeric))) sort(round(sapply(answer8[, sapply(answer8, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "d9d10")), "64c2e5a8b7bfe8730f31e34b3a39bb54"))
stopifnot("values in one or more character columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.character))) sum(sapply(answer8[sapply(answer8, is.character)], function(x) length(unique(x)))) else 0), "d9d10")), "b472483b0f89b5d33fdd3fbae96d7497"))
stopifnot("values in one or more factor columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.factor))) sum(sapply(answer8[, sapply(answer8, is.factor)], function(col) length(unique(col)))) else 0), "d9d10")), "c14625875d2365dcef80af1941b5163c"))

print('Success!')

### PART 3: Parquet

Apache Parquet is a free, open-source file format designed to store and process large amounts of data efficiently. It organizes data by column instead of by row, which can make storage smaller and speed up analysis. Parquet is useful when you’re working with complex structures like lists and nested records or large datasets, especially in situations where you’re only interested in a few columns at a time. Since the format stores data column by column, it can compress better and skip over unneeded data, reducing computational time needed. We will use the `arrow` package to work with Apache Parquet files in R. 

In this section, we will write the data scraped in `answer8` to Parquet files and then read them.

Let's create a new column with the year for each data point.

In [ ]:
cad_parquet_data <- answer8 |> 
  mutate(year = year(date))

cad_parquet_data |> group_by(year) |>  summarise(count = n())

**Question 9**
<br> (1 point)

Write `cad_parquet_data` into a set of parquet files based on the year. 

> **HINT**: use `group_by` and `write_dataset`.

*Replace the ... in the code chunk below with appropriate code. Uncomment the lines corresponding to code and run it. Assign the list to `answer9`.* 

In [ ]:
pq_path <- "./pq_files/"

if (!dir.exists(pq_path)) {
  dir.create(pq_path)
}

# cad_parquet_data |> 
#     ...(year) |> 
#     ...(..., format = ...)

# answer9 <- list.files(..., full.names = TRUE)


# YOUR CODE HERE
fail()


In [ ]:
library(digest)
stopifnot("type of answer9 is not character"= setequal(digest(paste(toString(class(answer9)), "178dc")), "d8383189e5a34763872319300f8badde"))
stopifnot("length of answer9 is not correct"= setequal(digest(paste(toString(length(answer9)), "178dc")), "f99a8797b901970a302d95fa73b77490"))
stopifnot("value of answer9 is not correct"= setequal(digest(paste(toString(tolower(answer9)), "178dc")), "7e5c34f5a9957fa8a403571529077142"))
stopifnot("letters in string value of answer9 are correct but case is not correct"= setequal(digest(paste(toString(answer9), "178dc")), "7e5c34f5a9957fa8a403571529077142"))

print('Success!')

**Question 10**
<br> (1 point)

Read the written parquet files and expand the files. 

> **HINT**: use `open_dataset()` and `collect()`.

*Store the expanded tibble in `answer10`.*  

In [ ]:
#answer10<-"FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer10)

In [ ]:
library(digest)
stopifnot("answer10 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer10)), "66bdc")), "15246fff0a24efa468bb05c9b135ddce"))
stopifnot("dimensions of answer10 are not correct"= setequal(digest(paste(toString(dim(answer10)), "66bdc")), "2cc0e1fc58a16c8eccd1ce151fbc0fbf"))
stopifnot("column names of answer10 are not correct"= setequal(digest(paste(toString(sort(colnames(answer10))), "66bdc")), "5101d877ce10250214fc6a84b7172e4d"))
stopifnot("types of columns in answer10 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer10, class)))), "66bdc")), "2f25b07b65fd2f298cda938a9d09417d"))
stopifnot("values in one or more numerical columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.numeric))) sort(round(sapply(answer10[, sapply(answer10, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "66bdc")), "826f869a9fdfd1bf72916186adca5ecc"))
stopifnot("values in one or more character columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.character))) sum(sapply(answer10[sapply(answer10, is.character)], function(x) length(unique(x)))) else 0), "66bdc")), "3d7f2ce0a2bd67ba5ee442449b8bdd2e"))
stopifnot("values in one or more factor columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.factor))) sum(sapply(answer10[, sapply(answer10, is.factor)], function(col) length(unique(col)))) else 0), "66bdc")), "7f47b0c090a8ea37cf5549eae9bc1ea1"))

print('Success!')